# Lab 3: Conditionals and Loops for Environmental Event Detection

        **Week:** Week 3

        **Lab type:** Individual lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Write conditionals.
- Use loops for classification.
- Count environmental events by month.
- Connect thresholds to scientific meaning.

        ## Earth and environmental motivation

        Environmental scientists often classify days as wet, dry, hot, cold, or high-flow before comparing events across seasons.

        ## Dataset

        Weather and streamflow processed CSV files

        ## Python concepts used

        - if/elif/else
- for loops
- Boolean logic
- Monthly grouping

## Lab 2 Debrief and Collaborative Debugging (First 20 Minutes)

Open the debrief card from Lab 2. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. The class will investigate one open problem one check at a time.

- 0-3 min: review the issue board.
- 3-11 min: student reports.
- 11-18 min: collaborative debugging.
- 18-20 min: record one reusable lesson and connect it to today's Lab.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

In [ ]:
import pandas as pd

weather = pd.read_csv(PROCESSED_DIR / "iowa_city_weather_daily.csv", parse_dates=["date"])
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])

In [ ]:
event_labels = []
for _, row in weather.iterrows():
    if row["precipitation_mm"] >= 10:
        label = "wet"
    elif row["temp_max_c"] >= 32:
        label = "hot"
    elif row["temp_min_c"] <= -10:
        label = "cold"
    else:
        label = "ordinary"
    event_labels.append(label)

weather["event_label"] = event_labels
print(weather["event_label"].value_counts())

In [ ]:
threshold = stream["discharge_cfs"].quantile(0.90)
stream["high_flow"] = stream["discharge_cfs"] >= threshold
stream["month"] = stream["date"].dt.month
monthly_high_flow = stream.groupby("month")["high_flow"].sum()
print(monthly_high_flow)

## Guided coding: how long was the longest dry spell?

Some questions need memory across loop steps. Here two counters track the
current run of dry days and the longest run seen so far.

In [ ]:
longest_dry = 0
current_dry = 0
for precip in weather["precipitation_mm"]:
    if precip < 1.0:
        current_dry = current_dry + 1
        if current_dry > longest_dry:
            longest_dry = current_dry
    else:
        current_dry = 0

print(f"Longest run of days with < 1 mm precipitation: {longest_dry} days")

## Guided coding: combined conditions, two ways

Muggy days need heat AND humidity at the same time. The loop version and the
vectorized version below must agree; comparing them is a free sanity check.

In [ ]:
muggy_loop = 0
for _, row in weather.iterrows():
    if row["temp_max_c"] >= 30 and row["relative_humidity_mean"] >= 70:
        muggy_loop = muggy_loop + 1

muggy_vectorized = int(((weather["temp_max_c"] >= 30) & (weather["relative_humidity_mean"] >= 70)).sum())
print("Muggy days, loop count:      ", muggy_loop)
print("Muggy days, vectorized count:", muggy_vectorized)

## Guided coding: a season function

Functions turn a rule into a reusable, testable unit. This one maps a month
number to a season name, and `.apply()` runs it on every row.

In [ ]:
def month_to_season(month):
    if month in (12, 1, 2):
        return "winter"
    elif month in (3, 4, 5):
        return "spring"
    elif month in (6, 7, 8):
        return "summer"
    else:
        return "fall"

weather["season"] = weather["date"].dt.month.apply(month_to_season)
wet_days_only = weather[weather["precipitation_mm"] >= 10]
print("Wet days (>= 10 mm) by season:")
print(wet_days_only["season"].value_counts())

## Try it yourself

Change one threshold and explain how the event counts change.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Write a function that counts days meeting both a temperature threshold and a precipitation threshold. Test it on a tiny DataFrame with an answer you can verify by hand, then apply it to the weather data using two different threshold pairs.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Count freeze-thaw days (minimum at or below 0 deg C AND maximum above
   0 deg C) for each month of the year. Which months have the most, and why
   does that matter for roads and rock weathering?
2. Recompute the high-flow days with a 95th-percentile threshold and compare
   the count with the 90th-percentile version.
3. In a markdown cell, state one threshold you would defend scientifically
   (for wet, hot, cold, or high flow) and give the reason in two sentences.

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Weather event counts
- [ ] Monthly high-flow counts
- [ ] One threshold sensitivity statement

        ## Short reflection

        How should a scientist choose an event threshold?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
